In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

print("Library berhasil di-import!")

Library berhasil di-import!


In [2]:
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

#menampilkan 5 baris pertama dari data train
train_df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
# Gabungkan sementara untuk pemrosesan yang konsisten
test_ids = test_df['PassengerId']
full_df = pd.concat([train_df, test_df], sort=False).reset_index(drop=True)

# 1. Ekstrak Title dari kolom Name
full_df['Title'] = full_df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)

# Pengelompokan gelar langka
title_mapping = {
    'Mr': 'Mr', 
    'Miss': 'Miss', 
    'Mrs': 'Mrs', 
    'Master': 'Master',
    'Col': 'Rare', 
    'Major': 'Rare', 
    'Rev': 'Rare', 
    'Sir': 'Rare',
    'Jonkheer': 'Rare', 
    'Don': 'Rare', 
    'Dona': 'Rare', 
    'Lady': 'Rare',
    'Countess': 'Rare', 
    'Capt': 'Rare', 
    'Dr': 'Rare', 
    'Mlle': 'Miss',
    'Mme': 'Mrs', 
    'Ms': 'Miss'
}
full_df['Title'] = full_df['Title'].map(title_mapping)

# 2. Buat fitur FamilySize dan IsAlone
full_df['FamilySize'] = full_df['SibSp'] + full_df['Parch'] + 1
full_df['IsAlone'] = (full_df['FamilySize'] == 1).astype(int)

# 3. Ekstrak huruf depan Cabin sebagai Dek
full_df['Deck'] = full_df['Cabin'].str[0].fillna('U')

print("Feature Engineering selesai!")

Feature Engineering selesai!


<>:6: SyntaxWarning: invalid escape sequence '\.'
<>:6: SyntaxWarning: invalid escape sequence '\.'
C:\Users\User\AppData\Local\Temp\ipykernel_28068\276377968.py:6: SyntaxWarning: invalid escape sequence '\.'
  full_df['Title'] = full_df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)


In [4]:
# Imputasi Age berdasarkan median kelompok Title dan Pclass
full_df['Age'] = full_df.groupby(['Title', 'Pclass'])['Age'].transform(lambda x: x.fillna(x.median()))

# Imputasi Fare berdasarkan median Pclass
full_df['Fare'] = full_df.groupby('Pclass')['Fare'].transform(lambda x: x.fillna(x.median()))

# Imputasi Embarked dengan modus ('S')
full_df['Embarked'] = full_df['Embarked'].fillna('S')

# Cek apakah masih ada missing values pada fitur utama
full_df[['Age', 'Fare', 'Embarked', 'Title']].isnull().sum()

Age         0
Fare        0
Embarked    0
Title       0
dtype: int64

In [5]:
# 1. One-Hot Encoding (tanpa dtype eksplisit agar kompatibel dengan versi pandas lama/baru)
categorical_cols = ['Sex', 'Embarked', 'Title', 'Deck']
full_df = pd.get_dummies(full_df, columns=categorical_cols, drop_first=True)

# Ubah kolom boolean hasil get_dummies menjadi integer (0/1) secara aman
cols_to_convert = [c for c in full_df.columns if full_df[c].dtype == 'bool']
full_df[cols_to_convert] = full_df[cols_to_convert].astype(int)

# 2. Pisahkan kembali menjadi train_clean dan test_clean
train_clean = full_df[full_df['Survived'].notnull()].copy()
test_clean = full_df[full_df['Survived'].isnull()].copy()

# 3. Tentukan kolom non-fitur yang ingin dibuang
drop_cols = ['PassengerId', 'Survived', 'Name', 'Ticket', 'Cabin']

# 4. Buat matriks X dan y menggunakan errors='ignore'
X = train_clean.drop(columns=drop_cols, errors='ignore')
y = train_clean['Survived'].astype(int)
X_test = test_clean.drop(columns=drop_cols, errors='ignore')

# 5. Penyesuaian struktur kolom train & test
X, X_test = X.align(X_test, join='left', axis=1, fill_value=0)

print(f"Cell 5 Berhasil! Jumlah Fitur Siap Pakai: {X.shape[1]}")

Cell 5 Berhasil! Jumlah Fitur Siap Pakai: 22


In [9]:
#Inisialisasi model Random Forest
model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)

#Evaluasi dengan 5-fold Stratified Cross-Validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X, y, cv=cv, scoring='accuracy')

print(f"Akurasi per lipatan (fold) : {np.round(scores * 100, 2)}")
print(f"Akurasi Rata-rata : {scores.mean() * 100:.2f}%")

#Latih model secara final ke seluruh data x dan y
model.fit(X, y)
print("Model berhasil dilatih dengan seluruh data train!")

Akurasi per lipatan (fold) : [83.8  82.58 82.58 82.58 84.83]
Akurasi Rata-rata : 83.28%
Model berhasil dilatih dengan seluruh data train!


In [ ]:
#Gunakan model untuk memprediksi data ujian
predictions = model.predict(X_test)

#Susun hasil prediksi ke dalam format DataFrame
submission = pd.DataFrame({
    'PassangerId' : test_ids,
    'Survived' : predicttions
})

#Simpan DataFrame menjadi file CSV tanpe menyimpan indeks baris
submission.to_csv('submission_titanic.csv', index=False)

print("File 'submission_titanic.csv' berhasil dibuat!")

#Tampilkan 5 baris pertama dari file tebakan
submission.head

SyntaxError: invalid syntax. Perhaps you forgot a comma? (2017467106.py, line 13)